In [3]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side
)
from openpyxl.utils import get_column_letter

raw_data = {
    'Month' : ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec'],
    'Revenue':  [4200000,3800000,4500000,5100000,
                 4800000,5400000,5200000,4900000,
                 5600000,6100000,5800000,6500000],
    'Expenses': [3100000,2900000,3300000,3600000,
                 3400000,3800000,3700000,3500000,
                 3900000,4200000,4000000,4400000],
    'Headcount':[45,45,47,47,48,50,50,51,52,52,53,55]
}

df = pd.DataFrame(raw_data)
print("Raw data loaded:")
print(df.head())

Raw data loaded:
  Month  Revenue  Expenses  Headcount
0   Jan  4200000   3100000         45
1   Feb  3800000   2900000         45
2   Mar  4500000   3300000         47
3   Apr  5100000   3600000         47
4   May  4800000   3400000         48


In [4]:
df['Net_Profit']       = df['Revenue'] - df['Expenses']
df['Profit_Margin_%']  = round(
    df['Net_Profit'] / df['Revenue'] * 100, 1)
df['Expense_Ratio_%']  = round(
    df['Expenses'] / df['Revenue'] * 100, 1)
df['Revenue_per_Head'] = round(
    df['Revenue'] / df['Headcount'], 0)
df['Revenue_MOM_%']    = round(
    df['Revenue'].pct_change() * 100, 1)
df['Profit_MOM_%']     = round(
    df['Net_Profit'].pct_change() * 100, 1)
df['Revenue_YTD']      = df['Revenue'].cumsum()
df['Net_Profit_YTD']   = df['Net_Profit'].cumsum()

print(df[['Month','Revenue','Net_Profit',
          'Profit_Margin_%','Revenue_MOM_%']])

   Month  Revenue  Net_Profit  Profit_Margin_%  Revenue_MOM_%
0    Jan  4200000     1100000             26.2            NaN
1    Feb  3800000      900000             23.7           -9.5
2    Mar  4500000     1200000             26.7           18.4
3    Apr  5100000     1500000             29.4           13.3
4    May  4800000     1400000             29.2           -5.9
5    Jun  5400000     1600000             29.6           12.5
6    Jul  5200000     1500000             28.8           -3.7
7    Aug  4900000     1400000             28.6           -5.8
8    Sep  5600000     1700000             30.4           14.3
9    Oct  6100000     1900000             31.1            8.9
10   Nov  5800000     1800000             31.0           -4.9
11   Dec  6500000     2100000             32.3           12.1


In [5]:
best_month    = df.loc[df['Net_Profit'].idxmax(),'Month']
worst_month   = df.loc[df['Net_Profit'].idxmin(),'Month']
avg_margin    = df['Profit_Margin_%'].mean()
total_revenue = df['Revenue'].sum()
total_profit  = df['Net_Profit'].sum()

print("=== KEY INSIGHTS ===")
print(f"Best month    : {best_month}")
print(f"Worst month   : {worst_month}")
print(f"Avg margin    : {avg_margin:.1f}%")
print(f"Total revenue : Rs {total_revenue:,.0f}")
print(f"Total profit  : Rs {total_profit:,.0f}")
print(f"Overall margin: {total_profit/total_revenue*100:.1f}%")

=== KEY INSIGHTS ===
Best month    : Dec
Worst month   : Feb
Avg margin    : 28.9%
Total revenue : Rs 61,900,000
Total profit  : Rs 18,100,000
Overall margin: 29.2%


In [6]:
wb = Workbook()
ws = wb.active
ws.title = 'MIS Report'

def make_border():
    s = Side(style='thin', color='D0D7DE')
    return Border(left=s,right=s,top=s,bottom=s)

def header_cell(cell, bg='1F3864'):
    cell.font      = Font(color='FFFFFF',bold=True,
                          name='Arial',size=10)
    cell.fill      = PatternFill('solid',start_color=bg)
    cell.alignment = Alignment(horizontal='center',
                                vertical='center')
    cell.border    = make_border()

def body_cell(cell,bold=False,color='000000',bg='FFFFFF'):
    cell.font      = Font(name='Arial',size=10,
                          bold=bold,color=color)
    cell.alignment = Alignment(horizontal='center',
                                vertical='center')
    cell.border    = make_border()
    cell.fill      = PatternFill('solid',start_color=bg)

ws.merge_cells('A1:J1')
ws['A1'] = 'ANNUAL FINANCIAL MIS REPORT — FY 2026'
ws['A1'].font = Font(color='FFFFFF',bold=True,
                     name='Arial',size=14)
ws['A1'].fill = PatternFill('solid',start_color='1E4D2B')
ws['A1'].alignment = Alignment(horizontal='center',
                                vertical='center')
ws.row_dimensions[1].height = 30

ws.merge_cells('A2:J2')
ws['A2'] = (f"Revenue: Rs {total_revenue:,.0f}  |  "
            f"Profit: Rs {total_profit:,.0f}  |  "
            f"Avg Margin: {avg_margin:.1f}%  |  "
            f"Best: {best_month}  |  Worst: {worst_month}")
ws['A2'].font = Font(color='FFFFFF',name='Arial',
                     size=9,italic=True)
ws['A2'].fill = PatternFill('solid',start_color='1E4D2B')
ws['A2'].alignment = Alignment(horizontal='center',
                                vertical='center')
ws.row_dimensions[2].height = 16

headers    = ['Month','Revenue','Expenses','Net Profit',
              'Margin %','Expense %','Rev/Head',
              'Rev MOM%','Profit MOM%','YTD Revenue']
col_widths = [10,14,14,14,11,11,13,12,13,16]

for ci,(h,w) in enumerate(zip(headers,col_widths),1):
    cell = ws.cell(row=3,column=ci,value=h)
    header_cell(cell)
    ws.column_dimensions[get_column_letter(ci)].width = w
ws.row_dimensions[3].height = 20

data_cols = ['Month','Revenue','Expenses','Net_Profit',
             'Profit_Margin_%','Expense_Ratio_%',
             'Revenue_per_Head','Revenue_MOM_%',
             'Profit_MOM_%','Revenue_YTD']

for ri,row in df.iterrows():
    excel_row = ri + 4
    bg = 'EBF3FB' if ri%2==0 else 'FFFFFF'
    ws.row_dimensions[excel_row].height = 17

    for ci,col in enumerate(data_cols,1):
        val = row[col]
        cell = ws.cell(row=excel_row,column=ci,value=val)

        if col in ['Revenue','Expenses','Net_Profit',
                   'Revenue_YTD','Revenue_per_Head']:
            cell.number_format = '#,##0'
        elif col in ['Profit_Margin_%','Expense_Ratio_%',
                     'Revenue_MOM_%','Profit_MOM_%']:
            cell.number_format = '0.0'

        if col in ['Revenue_MOM_%','Profit_MOM_%',
                   'Profit_Margin_%']:
            if pd.notna(val) and val > 0:
                body_cell(cell,color='22A05A',bg=bg)
            elif pd.notna(val) and val < 0:
                body_cell(cell,color='C0392B',bg=bg)
            else:
                body_cell(cell,bg=bg)
        elif ci == 1:
            body_cell(cell,bold=True,bg=bg)
        else:
            body_cell(cell,bg=bg)

tr = len(df) + 4
ws.row_dimensions[tr].height = 20
total_bg = 'D6EAF8'
ws.cell(row=tr,column=1,value='TOTAL')
body_cell(ws.cell(row=tr,column=1),bold=True,bg=total_bg)

for ci in [2,3,4]:
    col_letter = get_column_letter(ci)
    formula = f'=SUM({col_letter}4:{col_letter}{tr-1})'
    ws.cell(row=tr,column=ci,value=formula)
    ws.cell(row=tr,column=ci).number_format = '#,##0'
    body_cell(ws.cell(row=tr,column=ci),
              bold=True,bg=total_bg)

ws.freeze_panes = 'A4'
wb.save('financial_mis_report.xlsx')
print("Report saved. Open it and check formatting.")

Report saved. Open it and check formatting.
